# Step 2: Extract Video Metadata and Comments

This notebook ingests metadata and comments of YouTube videos retrieved with PYG lean, from folders specified in "config/data_config.yml", combining these data with categorization data where available.

It returns a csv table of the data for both metadata and comments data for each data collection, enriching the metadata with category data where available. The files are stored in the work directory specified in the dataset config yaml.


In [ ]:
import json
import pandas as pd
import csv
import yaml
from glob import glob
import os
import zipfile
from collections import defaultdict
import pprint

In [ ]:
# flags

dataset_config = "./config/dataset_config.yml"
catfile = '*combined_cleaned_data.csv'
output = './output/'
print(os.listdir(output))

In [ ]:
target_columns = [
    "Content match (yes no)", "Language Match (yes no)",
    "Main Video Source (choose 1)", "Formal Elements (multiple possible)",
    "Other Significant Tags (please describe new tags in the tag description column)",
    "YouTube Shorts (yes no)", "Other Noteworthy Formal Elements",
    "Content Type (choose 0-1)", "Content Focus (choose 0-2)",
    "Other Noteworthy Content Elements", "Values (Subjective Evaluation) muliple possible",
    "Other Noteworthy Evaluation", "new tags and other memos"
]
languagecodestoreplace = {'ko': 'ko', 
                          'en': 'en', 
                          'hant': 'zh-hant',
                          'zh-Hant': 'zh-hant', 
                          'jp': 'ja', 
                          'ja': 'ja', 
                          'hans': 'zh-hans',
                          'zh-Hans': 'zh-hans'}

In [ ]:
# data screening

print(os.path.abspath(dataset_config))
print(os.path.getsize(dataset_config))
with open(dataset_config, "r") as f:
    config = yaml.safe_load(f)
# Expand paths relative to working dir
directories = {
    key: {
        'wd': (
            [os.path.join(os.getcwd(), path) for path in value['wd']]
            if value['wd'] != None
            else []
        ),
        'catdir': (
            os.path.join(os.getcwd(), value['catdir'])
            if value['catdir'] != None
            else ''
        ),
    }
    for key, value in config.items()
}
for game in directories:
    tmpcatf =  glob(os.path.join(directories[game]['catdir'], catfile))
    if tmpcatf:
        directories[game]['catfile'] = tmpcatf[0]
        print(f'catfile detected: {directories[game]['catfile']}')
        directories[game]['catdata'] = pd.read_csv(directories[game]['catfile'])
    else:
        print('no category file exists')
        directories[game]['catfile'] = False

pprint.pprint(directories)

In [ ]:
def loadcatfile(catf):
    catdf = pd.read_csv(catf, lineterminator='\n')
    catdf['videoId'] = catdf['videoId'].str.strip()
    print('before deduplication')
    print(catdf['videoSearchRegion'].value_counts())
    for lang in ['zh-hans', 'zh-hant']:
        subset = catdf[catdf['videoSearchRegion'] == lang]
        print(f"\n{lang} shape: {subset.shape}")
        print(subset.isna().mean().sort_values())  # percent of NaNs per column
    catdf = catdf.drop_duplicates(subset=['videoId', 'videoSearchRegion'])
    print('after deduplication')
    print(catdf['videoSearchRegion'].value_counts())
    for lang in ['zh-hans', 'zh-hant']:
        subset = catdf[catdf['videoSearchRegion'] == lang]
        print(f"\n{lang} shape: {subset.shape}")
        print(subset.isna().mean().sort_values())  # percent of NaNs per column
    dupes = catdf.duplicated(subset=['videoId', 'videoSearchRegion'], keep=False)
    print(f"Number of duplicated entries: {dupes.sum()}")
    print(catdf[dupes].sort_values(['videoId', 'videoSearchRegion']))
    return catdf

In [ ]:
def extractmetadata(zip_file, blacklist, language):
    result = {}
    json_data = json.loads(zip_file.read('video_ids.json'))
        #print(json_data)
    idlist = {key: {} for key in json_data}
    nodata = 0
    for id in idlist:
        if id in blacklist:
            print(f'excluding {id}')
            return False
        metadata = {}
        try:
            #print(f'video_meta/{id}.json')
            json_data = json.loads(zip_file.read(f'video_meta/{id}.json'))
            #print(json_data)
            if len(json_data['items'])!=1:
                print(f'ERROR in video metadata for {id}: number of items {len(json_data["items"])}')
            else:
                metadata['videoId'] = id
                metadata['videoSearchRegion'] = language
                metadata['publishedAt'] = json_data['items'][0]['snippet']['publishedAt']
                metadata['channelId'] = json_data['items'][0]['snippet']['channelId']
                metadata['title'] = json_data['items'][0]['snippet']['title']
                metadata['description'] = json_data['items'][0]['snippet']['description']
                metadata['channelTitle'] = json_data['items'][0]['snippet']['channelTitle']
                try:
                    metadata['tags'] = json_data['items'][0]['snippet']['tags']
                except KeyError:
                    metadata['tags'] = 'missing'
                metadata['categoryId'] = json_data['items'][0]['snippet']['categoryId']
                metadata['liveBroadcastContent'] = json_data['items'][0]['snippet']['liveBroadcastContent']
                try:
                    metadata['defaultLanguage'] = json_data['items'][0]['snippet']['defaultLanguage']
                except KeyError:
                    metadata['defaultLanguage'] = 'missing'
                try:
                    metadata['defaultAudioLanguage'] = json_data['items'][0]['snippet']['defaultAudioLanguage']
                except KeyError:
                    metadata['defaultAudioLanguage'] = 'missing'
                metadata['duration'] = json_data['items'][0]['contentDetails']['duration']
                metadata['dimension'] = json_data['items'][0]['contentDetails']['dimension']
                metadata['definition'] = json_data['items'][0]['contentDetails']['definition']
                metadata['caption'] = json_data['items'][0]['contentDetails']['caption']
                metadata['licensedContent'] = json_data['items'][0]['contentDetails']['licensedContent']
                metadata['contentRating'] = json_data['items'][0]['contentDetails']['contentRating']
                metadata['projection'] = json_data['items'][0]['contentDetails']['projection']
                metadata['uploadStatus'] = json_data['items'][0]['status']['uploadStatus']
                metadata['privacyStatus'] = json_data['items'][0]['status']['privacyStatus']
                metadata['license'] = json_data['items'][0]['status']['license']
                metadata['embeddable'] = json_data['items'][0]['status']['embeddable']
                metadata['publicStatsViewable'] = json_data['items'][0]['status']['publicStatsViewable']
                metadata['madeForKids'] = json_data['items'][0]['status']['madeForKids']
                metadata['viewCount'] = json_data['items'][0]['statistics']['viewCount']
                try:
                    metadata['likeCount'] = json_data['items'][0]['statistics']['likeCount']
                except KeyError:
                    metadata['likeCount'] = 'missing'
                try:
                    metadata['dislikeCount'] = json_data['items'][0]['statistics']['dislikeCount']
                except KeyError:
                    metadata['dislikeCount'] = 'missing'
                metadata['favoriteCount'] = json_data['items'][0]['statistics']['favoriteCount']
                try:
                    metadata['commentCount'] = json_data['items'][0]['statistics']['commentCount']
                except KeyError:
                    metadata['commentCount'] = 'missing'
                try:
                    metadata['topicCategories'] = json_data['items'][0]['topicDetails']['topicCategories']
                except KeyError:
                    metadata['topicCategories'] = 'missing'
                #print(metadata)

                # add categorization result


                result[id] = metadata
        except KeyError:
            nodata += 1
            #print('no metadata found for ', id)
    print('no metadata found for ', nodata)
    return(pd.DataFrame.from_dict(result, orient='index'))

In [ ]:
def extractcomments(zip_file, blacklist, language):
    json_data = json.loads(zip_file.read('video_ids.json'))
    #print(json_data)
    idlist = {key: {} for key in json_data}
    comments = {}
    nodata = 0
    for id in idlist:
        if id in blacklist:
            print(f'excluding {id}')
            return False
        else:
            #print(comments)
            try:
                #print(f'video_comments/{id}_threads.json')
                json_data = json.loads(zip_file.read(f'video_comments/{id}_threads.json'))
                #print(json_data)
                for comment in json_data:
                    comment_dict = {}
                    comment_dict['commentId'] = comment['id']
                    comment_dict['videoId'] = id
                    comment_dict['videoSearchRegion'] = language
                    #print(comment['id'])
                    #comment_dict['textDisplay'] = comment['snippet']['topLevelComment']['snippet']['textDisplay']
                    comment_dict['textOriginal'] = comment['snippet']['topLevelComment']['snippet']['textOriginal']
                    comment_dict['authorDisplayName'] = comment['snippet']['topLevelComment']['snippet']['authorDisplayName']
                    #comment_dict['authorProfileImageUrl'] = comment['snippet']['topLevelComment']['snippet']['authorProfileImageUrl']
                    #comment_dict['authorChannelUrl'] = comment['snippet']['topLevelComment']['snippet']['authorChannelUrl']
                    try:
                        comment_dict['authorChannelId'] = comment['snippet']['topLevelComment']['snippet']['authorChannelId']['value']
                    except KeyError:
                        comment_dict['authorChannelId'] = 'missing'
                    #comment_dict['canRate'] = comment['snippet']['topLevelComment']['snippet']['canRate']
                    #comment_dict['viewerRating'] = comment['snippet']['topLevelComment']['snippet']['viewerRating']
                    comment_dict['likeCount'] = comment['snippet']['topLevelComment']['snippet']['likeCount']
                    #comment_dict['publishedAt'] = comment['snippet']['topLevelComment']['snippet']['publishedAt']
                    comment_dict['updatedAt'] = comment['snippet']['topLevelComment']['snippet']['updatedAt']
                    #comment_dict['canReply'] = comment['snippet']['canReply']
                    comment_dict['totalReplyCount'] = comment['snippet']['totalReplyCount']
                    #comment_dict['isPublic'] = comment['snippet']['isPublic']
                    comments[comment_dict['commentId']] = comment_dict
            except KeyError:
                nodata += 1
                #print('no comments data found for ', id)
#            print(comments.keys())
    print('no comments found for ', nodata)
    return(pd.DataFrame.from_dict(comments, orient='index'))

In [ ]:
def enrich_metadata_with_categories(df, catf):
    print('data enrichment...')
    print('length of dataframe', len(df))

    # Step 1: Exact match on both videoId and searchLanguage
    result = pd.merge(df, catf, on=['videoId', 'videoSearchRegion'], how='left', suffixes=('', '_cat'))

    # Step 2: Identify rows with no match
    unmatched = result[result['Main Video Source (choose 1)'].isna()][['videoId']].drop_duplicates()
    print(f"Primary match missing for {len(unmatched)} items, attempting fallback match by videoId only...")

    # Step 3: Fallback match only by videoId
    fallback = pd.merge(unmatched, catf.drop_duplicates('videoId'), on='videoId', how='left')

    # Step 4: Merge fallback results into result DataFrame
    result = pd.merge(
        result,
        fallback,
        on='videoId',
        how='left',
        suffixes=('', '_fallback')
    )

    # Step 5: Fill missing values from fallback columns
    for col in catf.columns:
        if col not in ['videoId', 'videoSearchRegion']:
            fallback_col = f"{col}_fallback"
            result[col] = result[col].combine_first(result[fallback_col])
            result.drop(columns=fallback_col, inplace=True)

    # Step 6: Report remaining unmatched
    still_missing = result[result['Main Video Source (choose 1)'].isna()]
    print(f"Remaining unmatched after fallback: {len(still_missing)} / {len(result)}")
    print(still_missing[['videoId', 'videoSearchRegion']].drop_duplicates())

    return result


In [ ]:
# Function to identify zip files in a directory and create a list of their filenames
def identify_zip_files_in_directory(directory):
    zip_file_list = []
    for file_name in os.listdir(directory):
        if file_name.endswith('.zip'):
            zip_file_list.append(file_name)
    return zip_file_list

In [ ]:
def createdatadfsfromdatasets(workdir, categoryfile):
    report = {}
    # Identify zip files in the directory and create a list of their filenames
    zip_files = identify_zip_files_in_directory(workdir)
    print(categoryfile)
    print(type(categoryfile))
    if categoryfile:
        catdata = loadcatfile(categoryfile)
    
    # Print the list of zip filenames
    print('list of files', zip_files)

#    tmp = ["Zelda_20231004.zip"]
    for dataset in zip_files:
        report[dataset] = {}
        print(dataset)
        date = dataset.split("_")[1].split(".")[0]
        report[dataset]['date'] = date
        with zipfile.ZipFile(os.path.join(workdir, dataset), 'r') as outer_zip:
            # Dictionary to store grouped filenames
            grouped_files = defaultdict(list)

            for filename in outer_zip.namelist():
                if '_None_' in filename:
                    key = filename.split('_None_')[0]
                    grouped_files[key].append(filename)
                elif '_nosource_' in filename:
                    key = filename.split('_nosource_')[0]
                    grouped_files[key].append(filename)
                else:
                    print('zip file content not meeting requirements', filename)

            # Convert to regular dict if needed
            grouped_files = dict(grouped_files)

            print(grouped_files)
            languages = [languagecodestoreplace[x.split('_')[1]] for x in grouped_files.keys()]
            for k, v in grouped_files.items():
                lang = k.split('_')[1]
                lang = languagecodestoreplace[lang]
                print('language identified as: ', lang)
                metadata = pd.DataFrame()
                comments = pd.DataFrame()
                for filename in v:
                    report[dataset][k] = {}
                    with outer_zip.open(filename) as inner_file:
                        with zipfile.ZipFile(inner_file) as inner_zip:
                            metadata = pd.concat([metadata, extractmetadata(inner_zip, [], lang)], ignore_index=True).drop_duplicates(subset=['videoId'])
                            metadata['videoId'] = metadata['videoId'].str.strip()
                            comments = pd.concat([comments, extractcomments(inner_zip, [], lang)], ignore_index=True).drop_duplicates(subset=['commentId'])
                report[dataset][k]['metadata_count'] = len(metadata)
                report[dataset][k]['comment_count'] = len(comments)
                if categoryfile:
                    metadata = enrich_metadata_with_categories(metadata, catdata)
                metadata.to_csv(os.path.join(workdir, f"ytma_{k}_metadata_cleaned.csv"))
                comments.to_csv(os.path.join(workdir, f"ytma_{k}_comments_cleaned.csv"))
        with open(os.path.join(workdir, f'ytma_{dataset}_report.json'), 'w+') as file:
            json.dump(report, file)
    print(f'languages in dataset: {languages}')
    print(f'duplicate detection: {grouped_files}')
    return report

# Execution section

In [ ]:
for game in directories.keys():
    for wd in directories[game]['wd']:
        createdatadfsfromdatasets(wd, directories[game]['catfile'])